# Study 10 — Clustering Robustness

In [ ]:
import os, sqlite3, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
import sqlite_vec

notebook_dir = Path.cwd()
while not (notebook_dir / 'data/db/pathosphere.db').exists():
    notebook_dir = notebook_dir.parent
    if notebook_dir == notebook_dir.parent: break
os.chdir(notebook_dir)

DB = Path('data/db/pathosphere.db').resolve()
CONN = sqlite3.connect(str(DB))
sqlite_vec.load(CONN)
CONN.row_factory = sqlite3.Row

In [ ]:
rss_events = CONN.execute('''
    SELECT e.id, e.title, COUNT(ed.document_id) as size
    FROM events e LEFT JOIN event_documents ed ON e.id = ed.event_id
    WHERE e.origin IN ("rss","comtrade") GROUP BY e.id ORDER BY size DESC
''').fetchall()

sizes = [e['size'] for e in rss_events]
print(f'Events: {len(rss_events)}, Singleton: {100*sum(1 for s in sizes if s==1)/len(sizes):.1f}%')

In [ ]:
all_docs = CONN.execute('''
    SELECT DISTINCT ed.document_id, ed.event_id
    FROM event_documents ed
    JOIN raw_documents r ON ed.document_id = r.id
    WHERE r.origin IN ("rss","comtrade")
''').fetchall()

doc_to_event = {d['document_id']: d['event_id'] for d in all_docs}

embeddings = []
valid_doc_ids = []
for doc_id in doc_to_event.keys():
    vec_row = CONN.execute('SELECT embedding FROM vec_documents WHERE document_id = ?', (doc_id,)).fetchone()
    if vec_row and vec_row['embedding']:
        emb = np.frombuffer(vec_row['embedding'], dtype=np.float32).copy()
        embeddings.append(emb)
        valid_doc_ids.append(doc_id)

embeddings = np.array(embeddings)
print(f'Embeddings: {len(embeddings)}')

In [ ]:
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

cluster_labels = np.array([doc_to_event[doc_id] for doc_id in valid_doc_ids])
unique_clusters = sorted(set(cluster_labels))
cluster_to_num = {cid: i for i, cid in enumerate(unique_clusters)}
cluster_nums = np.array([cluster_to_num[cid] for cid in cluster_labels])

fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=cluster_nums, cmap='tab20', s=50, alpha=0.6)

for event in rss_events[:10]:
    idx = [i for i, cid in enumerate(cluster_labels) if cid == event['id']]
    if idx:
        centroid = emb_2d[idx].mean(axis=0)
        ax.annotate(f"{event['size']}d", xy=centroid, fontsize=8, ha='center',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA Embedding Space — dots=docs, color=cluster, labels=top 10 by size')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Cluster ID')
ax.grid(True, alpha=0.3)
plt.savefig('study_10_embedding_space.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(sizes, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
ax1.set_xlabel('Docs per Cluster')
ax1.set_ylabel('Count')
ax1.set_title('Size Distribution (Linear)')
ax1.grid(alpha=0.3, axis='y')

ax2.hist(sizes, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
ax2.set_xlabel('Docs per Cluster')
ax2.set_ylabel('Count')
ax2.set_yscale('log')
ax2.set_title('Size Distribution (Log) — check bimodality')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('study_10_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
ax.axis('tight')
ax.axis('off')

table_data = []
for i, e in enumerate(rss_events[:10], 1):
    garbage = '❌ GARBAGE' if (e['title'] and e['title'].startswith('||')) else '✓ OK'
    table_data.append([str(i), f"{e['size']}", garbage, e['title'][:90]])

table = ax.table(cellText=table_data, colLabels=['#', 'Docs', 'Quality', 'Title'],
                cellLoc='left', loc='center', colWidths=[0.05, 0.08, 0.12, 0.75])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

for i in range(1, len(table_data) + 1):
    table[(i, 2)].set_facecolor('#ffcccc' if '❌' in table_data[i-1][2] else '#ccffcc')

plt.title('Top 10 Clusters', fontsize=14, pad=20)
plt.savefig('study_10_top_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
large = [e for e in rss_events if e['size'] >= 5][:30]
coh_data = []

for event in large:
    doc_ids_in = CONN.execute(
        'SELECT DISTINCT ed.document_id FROM event_documents ed WHERE ed.event_id = ?',
        (event['id'],)).fetchall()
    
    cluster_embs = []
    for d in doc_ids_in:
        if d['document_id'] in valid_doc_ids:
            idx = valid_doc_ids.index(d['document_id'])
            cluster_embs.append(embeddings[idx])
    
    if len(cluster_embs) >= 2:
        cluster_embs = np.array(cluster_embs)
        centroid = cluster_embs.mean(axis=0)
        dists = [np.linalg.norm(e - centroid) for e in cluster_embs]
        sim = 1 - np.mean(dists)**2 / 2
        coh_data.append({'size': event['size'], 'coh': sim, 'title': event['title'][:25]})

df = pd.DataFrame(coh_data).sort_values('coh')
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['green' if c >= 0.88 else 'orange' for c in df['coh']]
ax.barh(range(len(df)), df['coh'], color=colors, alpha=0.7)
ax.axvline(0.88, color='red', linestyle='--', linewidth=2, label='Threshold')
ax.set_yticks(range(len(df)))
ax.set_yticklabels([f"{s}d: {t}" for s, t in zip(df['size'], df['title'])], fontsize=9)
ax.set_xlabel('Avg Similarity to Centroid')
ax.set_title('Cluster Coherence')
ax.legend()
ax.grid(alpha=0.3, axis='x')
plt.savefig('study_10_coherence.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean coherence: {df["coh"].mean():.3f}')

In [ ]:
garbage = CONN.execute("SELECT COUNT(*) FROM events WHERE title LIKE '||%'").fetchone()[0]
singleton = 100*sum(1 for s in sizes if s==1)/len(sizes)

print(f'Events: {len(rss_events)}')
print(f'Singleton: {singleton:.1f}%')
print(f'Mean size: {np.mean(sizes):.2f}')
print(f'Garbage: {garbage}')
print(f'Status: {"SOLID" if garbage==0 else "BROKEN"}')